# DS317_02 - Khám Phá, Kiểm Kê & Đo Lường CLDL (RAW 18 vs PREP27)

Notebook kiểm toán toàn diện hai phiên bản dữ liệu NOAA AIS 2025:
1. **Bản thô (`RAW` - 18 cột):** `01_ban_tho_18_thuoc_tinh/{train,test}.parquet` (hoặc `q1..q4.parquet` / tùy chọn `noaa_ais_2025_raw_labeled.parquet`).
2. **Bản 26 đặc trưng + nhãn (`PREP27` - 27 cột):** `03_ban_tien_xu_ly_27_thuoc_tinh/{train,test}.parquet` (hoặc `q1..q4.parquet`), gồm 4 cột định danh/thời gian, 2 góc thô, 20 đặc trưng mô hình và 1 cột `Target_Hours`.

## Cấu Trúc Thực Thi
- **C01-C02:** Cấu hình, tự động định vị đầu vào và kiểm tra schema 18/27 cột.
- **Phần A (C03-C05):** Quét streaming theo batch (`BATCH_SIZE = 500_000`), tính Welford moments mọi cột và đo 6 chiều DAMA-DMBOK đồng nhất Trước vs Sau.
- **Phần B (C06-C08):** EDA phân phối `Target_Hours` đầy đủ (không cắt 168h), tương quan Pearson & Spearman và 5 biểu đồ đối sánh.
- **Phần C (C09-C12):** Kiểm chứng Mask-and-Recover 3 cấp, đối sánh nhanh LightGBM (`RAW` vs `PREP27`) trên tầng `All / LE_168h / GT_168h` và đóng gói ZIP.

In [ ]:
"""C01: Cau hinh moi truong, tham so quet va thu muc ket qua."""

# 1. Standard library imports
from collections import Counter, deque
import gc
import importlib.util
import json
import os
from pathlib import Path
import platform
import time
from typing import Any, Dict, List, Optional, Tuple
import zipfile

# 2. Third-party imports
from IPython.display import FileLink, display
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.stats import spearmanr

REQUIRED_PKGS = ['pyarrow', 'lightgbm', 'scipy', 'matplotlib']
MISSING_PKGS = [
    pkg for pkg in REQUIRED_PKGS if importlib.util.find_spec(pkg) is None
]
if MISSING_PKGS:
    raise ImportError(
        'Thieu thu vien bat buoc: ' + ', '.join(MISSING_PKGS)
    )

ROOT = Path(
    os.environ.get(
        'DS317_ROOT',
        '/kaggle/input/noaa-ais-2025-full-year-tin-x-l-nn-tng',
    )
)
OUT = Path(os.environ.get('DS317_OUT', '/kaggle/working/ds317_02_results'))
BATCH_SIZE = int(os.environ.get('DS317_BATCH_SIZE', '500000'))
SAMPLE_SIZE = int(os.environ.get('DS317_SAMPLE_SIZE', '320000'))
SEED = int(os.environ.get('DS317_SEED', '42'))
MAX_BATCHES_RAW = os.environ.get('DS317_MAX_BATCHES', '')
MAX_BATCHES: Optional[int] = (
    int(MAX_BATCHES_RAW) if MAX_BATCHES_RAW.strip() else None
)
DEDUP_BUFFER_CAP = 500_000
N_JOBS = max(1, min(4, os.cpu_count() or 2))

SUB_DIRS = [
    'COMMON',
    'DICTIONARY',
    'DAMA',
    'EDA',
    'CORRELATION',
    'VALIDATION',
    'CHARTS',
]
for sub_name in SUB_DIRS:
    (OUT / sub_name).mkdir(parents=True, exist_ok=True)

COMMON = OUT / 'COMMON'
DICT_DIR = OUT / 'DICTIONARY'
DAMA_DIR = OUT / 'DAMA'
EDA_DIR = OUT / 'EDA'
CORR_DIR = OUT / 'CORRELATION'
VAL_DIR = OUT / 'VALIDATION'
CHART_DIR = OUT / 'CHARTS'

print(
    f'[C01 CONFIG] ROOT={ROOT} | OUT={OUT} | '
    f'BATCH_SIZE={BATCH_SIZE:,} | SAMPLE_SIZE={SAMPLE_SIZE:,}'
)

In [ ]:
"""C02: Dinh vi linh hoat dau vao RAW (18 cot) & PREP27 (27 cot)."""

COLS_RAW: List[str] = [
    'mmsi', 'trip_id', 'base_date_time', 'longitude', 'latitude',
    'sog', 'cog', 'heading', 'status', 'vessel_type', 'length',
    'width', 'draft', 'cargo', 'berth_lon', 'berth_lat', 'ata_time',
    'Target_Hours',
]

COLS_PREP27: List[str] = [
    'mmsi', 'trip_id', 'base_date_time', 'longitude', 'latitude',
    'sog', 'cog', 'heading', 'cog_sin', 'cog_cos', 'heading_sin',
    'heading_cos', 'status', 'vessel_type', 'length', 'width',
    'draft', 'cargo', 'berth_lon', 'berth_lat', 'ata_time',
    'Dist_To_Berth_NM', 'Area_Proxy', 'Volume_Proxy', 'Is_Tanker',
    'imputation_flag', 'Target_Hours',
]

ID_TIME_COLS = ['mmsi', 'trip_id', 'base_date_time', 'ata_time']
RAW_ANGLE_COLS = ['cog', 'heading']
TARGET_COL = 'Target_Hours'
MODEL_FEATURES_20 = [
    c for c in COLS_PREP27
    if c not in ID_TIME_COLS + RAW_ANGLE_COLS + [TARGET_COL]
]
RAW_MODEL_FEATURES_13 = [
    'longitude', 'latitude', 'sog', 'cog', 'heading', 'status',
    'vessel_type', 'length', 'width', 'draft', 'cargo',
    'berth_lon', 'berth_lat',
]


def _find_version_files(
    version_dir_name: str,
    allow_presplit_raw: bool = False,
) -> Tuple[str, List[Tuple[str, Path]]]:
    """Tim bo parquet cho 1 phien ban tren ROOT hoac /kaggle/input."""
    search_bases = [ROOT] if ROOT.exists() else []
    kaggle_in = Path('/kaggle/input')
    if kaggle_in.exists() and kaggle_in not in search_bases:
        search_bases.append(kaggle_in)

    cand_dirs: List[Path] = []
    for base in search_bases:
        direct = base / version_dir_name
        if direct.is_dir():
            cand_dirs.append(direct)
        for hit in sorted(base.rglob(version_dir_name)):
            if hit.is_dir() and hit not in cand_dirs:
                cand_dirs.append(hit)

    for vdir in cand_dirs:
        tr_p, te_p = vdir / 'train.parquet', vdir / 'test.parquet'
        if tr_p.is_file() and te_p.is_file():
            return 'SPLIT_POST', [('train', tr_p), ('test', te_p)]
        q_paths = [
            (q, vdir / f'{q}.parquet') for q in ('q1', 'q2', 'q3', 'q4')
        ]
        if all(p.is_file() for _, p in q_paths):
            q_files = [
                ('train' if q in ('q1', 'q2', 'q3') else 'test', p)
                for q, p in q_paths
            ]
            return 'SPLIT_POST_QUARTERS', q_files

    if allow_presplit_raw:
        for base in search_bases:
            hits = sorted(base.rglob('noaa_ais_2025_raw_labeled.parquet'))
            if hits:
                return 'PRE_SPLIT', [('pre_split', hits[0])]

    raise FileNotFoundError(
        f'Khong tim thay du lieu cho {version_dir_name} trong {search_bases}'
    )


def _check_schema(files: List[Tuple[str, Path]], req_cols: List[str]) -> int:
    """Xac minh schema va tra ve tong so dong metadata."""
    total_rows = 0
    for _, fpath in files:
        pf = pq.ParquetFile(fpath)
        names = pf.schema_arrow.names
        missing = [c for c in req_cols if c not in names]
        if missing:
            raise ValueError(f'{fpath} thieu cot bat buoc: {missing}')
        total_rows += int(pf.metadata.num_rows)
        del pf
    return total_rows


RAW_MODE, RAW_FILES = _find_version_files(
    '01_ban_tho_18_thuoc_tinh', allow_presplit_raw=True
)
PREP_MODE, PREP_FILES = _find_version_files(
    '03_ban_tien_xu_ly_27_thuoc_tinh', allow_presplit_raw=False
)

RAW_META_ROWS = _check_schema(RAW_FILES, COLS_RAW)
PREP_META_ROWS = _check_schema(PREP_FILES, COLS_PREP27)

det_rows = []
for ver, mode, files in [
    ('RAW_18', RAW_MODE, RAW_FILES),
    ('PREP_27', PREP_MODE, PREP_FILES),
]:
    for split_lbl, fpath in files:
        pf = pq.ParquetFile(fpath)
        det_rows.append({
            'version': ver,
            'input_mode': mode,
            'split_tag': split_lbl,
            'file_path': str(fpath),
            'num_rows': int(pf.metadata.num_rows),
            'num_cols': len(pf.schema_arrow.names),
            'size_mb': round(fpath.stat().st_size / (1024 ** 2), 2),
        })
        del pf
gc.collect()
df_detected = pd.DataFrame(det_rows)
df_detected.to_csv(COMMON / 'detected_inputs.csv', index=False)

print(f'[RAW] mode={RAW_MODE} | rows={RAW_META_ROWS:,}')
print(f'[PREP27] mode={PREP_MODE} | rows={PREP_META_ROWS:,}')
print(
    f'[SCHEMA PREP27] 27 cols = 4 ID/Time + 2 RawAngle + '
    f'{len(MODEL_FEATURES_20)} ModelFeatures + 1 Target_Hours'
)
print('ModelFeatures(20):', MODEL_FEATURES_20)
display(df_detected)

## Phần A: Động Cơ Quét Streaming & Đo Lường 6 Chiều DAMA-DMBOK Đồng Nhất

- `WelfordTracker`: tích lũy chính xác `min`, `max`, `mean`, `std`, tỷ lệ `NaN`, `Inf`, `0`, và số âm trên toàn bộ 100% số dòng.
- `StreamingPearsonTracker`: cập nhật trực tuyến hiệp phương sai theo công thức Chan/Pébay để tính chính xác `Pearson r(Dist_To_Berth_NM, Target_Hours)`.
- `SlidingBatchDeduplicator`: phát hiện trùng lặp khóa `(mmsi, base_date_time)` cả bên trong batch và xuyên ranh giới batch.
- `evaluate_dama_batch(...)`: áp dụng **cùng một bộ quy tắc** cho cả `RAW` và `PREP27`, tách bạch đầy đủ `Completeness` (gồm/không gồm `cargo`), `Consistency_Operational` (`<= 168h`) vs `Consistency_Kinematic` (không cắt 168h), `Accuracy`, `Timeliness` (`2025 Causal` vs `168h Window`), `Validity` và `Uniqueness`.

In [ ]:
"""C04: Dinh nghia lop streaming Welford, Deduplicator va ham DAMA."""


class WelfordTracker:
    """Bo dem thong ke streaming theo thuat toan Chan/Welford."""

    def __init__(self, col_name: str) -> None:
        self.col_name = col_name
        self.total = 0
        self.n_nan = 0
        self.n_inf = 0
        self.n_valid = 0
        self.n_zero = 0
        self.n_neg = 0
        self.mean = 0.0
        self.m2 = 0.0
        self.v_min = np.inf
        self.v_max = -np.inf

    def update(self, arr: np.ndarray) -> None:
        """Cap nhat tracker voi 1 mang so numpy."""
        n_all = len(arr)
        if n_all == 0:
            return
        self.total += n_all
        nan_mask = np.isnan(arr)
        inf_mask = np.isinf(arr)
        self.n_nan += int(nan_mask.sum())
        self.n_inf += int(inf_mask.sum())
        valid = arr[~(nan_mask | inf_mask)].astype(np.float64, copy=False)
        n_b = len(valid)
        if n_b == 0:
            return
        self.n_zero += int((valid == 0.0).sum())
        self.n_neg += int((valid < 0.0).sum())
        b_min = float(valid.min())
        b_max = float(valid.max())
        if b_min < self.v_min:
            self.v_min = b_min
        if b_max > self.v_max:
            self.v_max = b_max
        b_mean = float(valid.mean())
        b_m2 = float(((valid - b_mean) ** 2).sum())
        if self.n_valid == 0:
            self.n_valid = n_b
            self.mean = b_mean
            self.m2 = b_m2
        else:
            n_tot = self.n_valid + n_b
            delta = b_mean - self.mean
            self.m2 += b_m2 + (delta ** 2) * (self.n_valid * n_b / n_tot)
            self.mean = (self.n_valid * self.mean + n_b * b_mean) / n_tot
            self.n_valid = n_tot

    def summary(self) -> Dict[str, Any]:
        """Tra ve tu dien tong hop cho 1 cot."""
        std = (
            float(np.sqrt(self.m2 / (self.n_valid - 1)))
            if self.n_valid > 1
            else np.nan
        )
        miss = self.n_nan + self.n_inf
        denom_v = max(self.n_valid, 1)
        return {
            'column': self.col_name,
            'total_rows': self.total,
            'non_null': self.n_valid,
            'missing_count': miss,
            'missing_pct': round(100.0 * miss / max(self.total, 1), 4),
            'nan_count': self.n_nan,
            'inf_count': self.n_inf,
            'zeros_pct': round(100.0 * self.n_zero / denom_v, 4),
            'negatives_pct': round(100.0 * self.n_neg / denom_v, 4),
            'min': None if self.n_valid == 0 else round(self.v_min, 6),
            'max': None if self.n_valid == 0 else round(self.v_max, 6),
            'mean': None if self.n_valid == 0 else round(self.mean, 6),
            'std': None if np.isnan(std) else round(std, 6),
        }


class StreamingPearsonTracker:
    """Tinh Pearson r streaming cho cap bien (x, y) khong tran so."""

    def __init__(self) -> None:
        self.n = 0
        self.mx = 0.0
        self.my = 0.0
        self.cxx = 0.0
        self.cyy = 0.0
        self.cxy = 0.0

    def update(self, x: np.ndarray, y: np.ndarray) -> None:
        """Cap nhat Pearson voi 1 batch cap mang (x, y)."""
        mask = np.isfinite(x) & np.isfinite(y)
        xb = x[mask].astype(np.float64, copy=False)
        yb = y[mask].astype(np.float64, copy=False)
        nb = len(xb)
        if nb == 0:
            return
        mxb = float(xb.mean())
        myb = float(yb.mean())
        dx_b = xb - mxb
        dy_b = yb - myb
        cxx_b = float((dx_b * dx_b).sum())
        cyy_b = float((dy_b * dy_b).sum())
        cxy_b = float((dx_b * dy_b).sum())
        if self.n == 0:
            self.n = nb
            self.mx, self.my = mxb, myb
            self.cxx, self.cyy, self.cxy = cxx_b, cyy_b, cxy_b
            return
        n_tot = self.n + nb
        dx = mxb - self.mx
        dy = myb - self.my
        wt = self.n * nb / n_tot
        self.cxx += cxx_b + dx * dx * wt
        self.cyy += cyy_b + dy * dy * wt
        self.cxy += cxy_b + dx * dy * wt
        self.mx = (self.n * self.mx + nb * mxb) / n_tot
        self.my = (self.n * self.my + nb * myb) / n_tot
        self.n = n_tot

    def correlation(self) -> float:
        """Tra ve he so Pearson r hien tai."""
        denom = np.sqrt(self.cxx * self.cyy)
        if self.n < 2 or denom <= 0.0:
            return float('nan')
        return float(self.cxy / denom)


class SlidingBatchDeduplicator:
    """Dem ban ghi trung lap (mmsi, base_date_time) xuyen batch."""

    def __init__(self, max_keys: int = DEDUP_BUFFER_CAP) -> None:
        self.max_keys = max_keys
        self.seen_set: set = set()
        self.fifo: deque = deque()

    def count_duplicates(
        self, mmsi_arr: np.ndarray, bdt_ns: np.ndarray
    ) -> int:
        """Dem so dong trung trong batch va voi cua so truot."""
        if len(mmsi_arr) == 0:
            return 0
        key_df = pd.DataFrame({'m': mmsi_arr, 't': bdt_ns})
        hashes = pd.util.hash_pandas_object(key_df, index=False).to_numpy()
        dup_in_batch = key_df.duplicated(keep='first').to_numpy()
        dups = int(dup_in_batch.sum())
        unique_hashes = hashes[~dup_in_batch]
        cross_dups = 0
        for h_val in unique_hashes.tolist():
            if h_val in self.seen_set:
                cross_dups += 1
            else:
                self.seen_set.add(h_val)
                self.fifo.append(h_val)
                if len(self.fifo) > self.max_keys:
                    old = self.fifo.popleft()
                    self.seen_set.discard(old)
        return dups + cross_dups


def compute_geodesic_dist_nm(df: pd.DataFrame) -> np.ndarray:
    """Tinh khoang cach geodesic (NM) dung cong thuc chuan DS317_01."""
    lon1 = pd.to_numeric(df['longitude'], errors='coerce').to_numpy(float)
    lat1 = pd.to_numeric(df['latitude'], errors='coerce').to_numpy(float)
    lon2 = pd.to_numeric(df['berth_lon'], errors='coerce').to_numpy(float)
    lat2 = pd.to_numeric(df['berth_lat'], errors='coerce').to_numpy(float)
    ok = (
        np.isfinite(lon1) & np.isfinite(lat1)
        & np.isfinite(lon2) & np.isfinite(lat2)
        & (lat1 >= -90.0) & (lat1 <= 90.0)
        & (lon1 >= -180.0) & (lon1 <= 180.0)
        & (lat2 >= -90.0) & (lat2 <= 90.0)
        & (lon2 >= -180.0) & (lon2 <= 180.0)
    )
    out = np.full(len(df), np.nan, dtype=np.float64)
    if not np.any(ok):
        return out
    r_lat1 = np.radians(lat1[ok])
    r_lat2 = np.radians(lat2[ok])
    dl = (np.radians(lon2[ok] - lon1[ok]) + np.pi) % (2.0 * np.pi) - np.pi
    hav = (
        np.sin((r_lat2 - r_lat1) / 2.0) ** 2
        + np.cos(r_lat1) * np.cos(r_lat2) * np.sin(dl / 2.0) ** 2
    )
    phi = (r_lat1 + r_lat2) / 2.0
    a_eq, b_po = 6378.137, 6356.752314245
    rad = np.sqrt(
        ((a_eq ** 2 * np.cos(phi)) ** 2 + (b_po ** 2 * np.sin(phi)) ** 2)
        / ((a_eq * np.cos(phi)) ** 2 + (b_po * np.sin(phi)) ** 2)
    )
    out[ok] = 2.0 * rad * np.arcsin(np.sqrt(np.clip(hav, 0.0, 1.0))) / 1.852
    return out


def _empty_dama_acc() -> Dict[str, Any]:
    """Khoi tao bo dem tich luy cho 6 chieu DAMA."""
    return {
        'rows': 0,
        'rec_comp_all': 0,
        'rec_comp_non_cargo': 0,
        'cell_nonnull_all': 0,
        'cell_total_all': 0,
        'cell_nonnull_non_cargo': 0,
        'cell_total_non_cargo': 0,
        'cons_kinematic': 0,
        'cons_operational': 0,
        'acc_kinematic': 0,
        'time_2025_causal': 0,
        'time_168h_window': 0,
        'validity': 0,
        'duplicates': 0,
        'pearson': StreamingPearsonTracker(),
    }


def evaluate_dama_batch(
    df: pd.DataFrame,
    dist_nm: np.ndarray,
    dedup: SlidingBatchDeduplicator,
    acc_map: Dict[str, Dict[str, Any]],
    group_keys: List[str],
) -> None:
    """Do 6 chieu DAMA dong nhat tren 1 batch va cong don vao acc_map."""
    n = len(df)
    if n == 0:
        return

    all_cols = list(df.columns)
    non_cargo_cols = [c for c in all_cols if c != 'cargo']
    na_mat_all = df[all_cols].isna().to_numpy()
    na_mat_nc = df[non_cargo_cols].isna().to_numpy()
    row_ok_all = ~na_mat_all.any(axis=1)
    row_ok_nc = ~na_mat_nc.any(axis=1)
    row_nn_all = len(all_cols) - na_mat_all.sum(axis=1)
    row_nn_nc = len(non_cargo_cols) - na_mat_nc.sum(axis=1)

    mmsi = pd.to_numeric(df['mmsi'], errors='coerce').to_numpy(np.float64)
    lat = pd.to_numeric(df['latitude'], errors='coerce').to_numpy(np.float64)
    lon = pd.to_numeric(df['longitude'], errors='coerce').to_numpy(np.float64)
    sog = pd.to_numeric(df['sog'], errors='coerce').to_numpy(np.float64)
    cog = pd.to_numeric(df['cog'], errors='coerce').to_numpy(np.float64)
    hdg = pd.to_numeric(df['heading'], errors='coerce').to_numpy(np.float64)
    st = pd.to_numeric(df['status'], errors='coerce').to_numpy(np.float64)
    vt = pd.to_numeric(df['vessel_type'], errors='coerce').to_numpy(np.float64)
    ln = pd.to_numeric(df['length'], errors='coerce').to_numpy(np.float64)
    wd = pd.to_numeric(df['width'], errors='coerce').to_numpy(np.float64)
    dr = pd.to_numeric(df['draft'], errors='coerce').to_numpy(np.float64)
    tgt = pd.to_numeric(
        df['Target_Hours'], errors='coerce'
    ).to_numpy(np.float64)

    bdt = pd.to_datetime(df['base_date_time'], errors='coerce')
    ata = pd.to_datetime(df['ata_time'], errors='coerce')
    bdt_ns = bdt.astype('datetime64[ns]').astype(np.int64).to_numpy()
    causal_ok = (bdt.notna() & ata.notna() & (bdt < ata)).to_numpy()
    yr_2025 = (bdt.dt.year == 2025).fillna(False).to_numpy()

    vt_int_ok = np.isfinite(vt) & (vt >= 70.0) & (vt <= 89.0) & (vt % 1 == 0)
    max_d = np.where(vt >= 80.0, 1800.0, 2300.0)
    dist_ok = np.isfinite(dist_nm) & (dist_nm >= 0.0) & (dist_nm <= max_d)
    st_ok = np.isfinite(st) & (st != 1.0) & (st != 5.0)
    sog_ok = np.isfinite(sog) & (sog >= 1.0) & (sog <= 40.0)
    tgt_pos = np.isfinite(tgt) & (tgt > 0.0)
    tgt_168 = tgt_pos & (tgt <= 168.0)

    cons_kin = sog_ok & st_ok & causal_ok & dist_ok
    cons_op = cons_kin & tgt_168
    imp_spd = np.where(tgt_pos, dist_nm / np.where(tgt_pos, tgt, 1.0), np.nan)
    acc_kin = dist_ok & tgt_pos & np.isfinite(imp_spd) & (imp_spd >= 1.62)
    t_2025_c = yr_2025 & causal_ok
    t_168_w = t_2025_c & tgt_168

    val_ok = (
        np.isfinite(mmsi) & (mmsi >= 100_000_000) & (mmsi <= 999_999_999)
        & np.isfinite(lat) & (lat >= 15.0) & (lat <= 75.0)
        & np.isfinite(lon) & (lon >= -180.0) & (lon <= -65.0)
        & sog_ok & st_ok & vt_int_ok
        & np.isfinite(ln) & (ln > 0.0)
        & np.isfinite(wd) & (wd > 0.0)
        & np.isfinite(dr) & (dr > 0.0) & (dr <= 25.5)
        & np.isfinite(cog) & (cog >= 0.0) & (cog < 360.0)
        & np.isfinite(hdg) & (hdg >= 0.0) & (hdg < 360.0)
        & tgt_pos
    )

    q_num = ((bdt.dt.month - 1) // 3 + 1).fillna(0).to_numpy(np.int64)
    dups_total = dedup.count_duplicates(mmsi, bdt_ns)

    def _accum(key: str, mask: Optional[np.ndarray], dup_est: int) -> None:
        if key not in acc_map:
            acc_map[key] = _empty_dama_acc()
        st_d = acc_map[key]
        if mask is None:
            st_d['rows'] += n
            st_d['rec_comp_all'] += int(row_ok_all.sum())
            st_d['rec_comp_non_cargo'] += int(row_ok_nc.sum())
            st_d['cell_nonnull_all'] += int(row_nn_all.sum())
            st_d['cell_total_all'] += n * len(all_cols)
            st_d['cell_nonnull_non_cargo'] += int(row_nn_nc.sum())
            st_d['cell_total_non_cargo'] += n * len(non_cargo_cols)
            st_d['cons_kinematic'] += int(cons_kin.sum())
            st_d['cons_operational'] += int(cons_op.sum())
            st_d['acc_kinematic'] += int(acc_kin.sum())
            st_d['time_2025_causal'] += int(t_2025_c.sum())
            st_d['time_168h_window'] += int(t_168_w.sum())
            st_d['validity'] += int(val_ok.sum())
            st_d['duplicates'] += dup_est
            st_d['pearson'].update(dist_nm, tgt)
            return
        cnt = int(mask.sum())
        if cnt == 0:
            return
        st_d['rows'] += cnt
        st_d['rec_comp_all'] += int(row_ok_all[mask].sum())
        st_d['rec_comp_non_cargo'] += int(row_ok_nc[mask].sum())
        st_d['cell_nonnull_all'] += int(row_nn_all[mask].sum())
        st_d['cell_total_all'] += cnt * len(all_cols)
        st_d['cell_nonnull_non_cargo'] += int(row_nn_nc[mask].sum())
        st_d['cell_total_non_cargo'] += cnt * len(non_cargo_cols)
        st_d['cons_kinematic'] += int(cons_kin[mask].sum())
        st_d['cons_operational'] += int(cons_op[mask].sum())
        st_d['acc_kinematic'] += int(acc_kin[mask].sum())
        st_d['time_2025_causal'] += int(t_2025_c[mask].sum())
        st_d['time_168h_window'] += int(t_168_w[mask].sum())
        st_d['validity'] += int(val_ok[mask].sum())
        st_d['duplicates'] += round(dups_total * (cnt / n))
        st_d['pearson'].update(dist_nm[mask], tgt[mask])

    for g_key in group_keys:
        _accum(g_key, None, dups_total)
    for q_i in (1, 2, 3, 4):
        q_mask = q_num == q_i
        if np.any(q_mask):
            _accum(f'Q{q_i}', q_mask, 0)


def finalize_dama_table(
    acc_map: Dict[str, Dict[str, Any]], version_tag: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Chuyen doi acc_map thanh 2 bang DAMA (Overall/Split va Q1-Q4)."""
    rows = []
    for scope, st_d in acc_map.items():
        n = max(st_d['rows'], 1)
        rows.append({
            'version': version_tag,
            'scope': scope,
            'rows': st_d['rows'],
            'completeness_rec_all_pct': round(
                100.0 * st_d['rec_comp_all'] / n, 4
            ),
            'completeness_rec_non_cargo_pct': round(
                100.0 * st_d['rec_comp_non_cargo'] / n, 4
            ),
            'completeness_field_all_pct': round(
                100.0 * st_d['cell_nonnull_all']
                / max(st_d['cell_total_all'], 1),
                4,
            ),
            'completeness_field_non_cargo_pct': round(
                100.0 * st_d['cell_nonnull_non_cargo']
                / max(st_d['cell_total_non_cargo'], 1),
                4,
            ),
            'consistency_kinematic_pct': round(
                100.0 * st_d['cons_kinematic'] / n, 4
            ),
            'consistency_operational_pct': round(
                100.0 * st_d['cons_operational'] / n, 4
            ),
            'accuracy_kinematic_pct': round(
                100.0 * st_d['acc_kinematic'] / n, 4
            ),
            'pearson_dist_target_r': round(
                st_d['pearson'].correlation(), 5
            ),
            'timeliness_2025_causal_pct': round(
                100.0 * st_d['time_2025_causal'] / n, 4
            ),
            'timeliness_168h_window_pct': round(
                100.0 * st_d['time_168h_window'] / n, 4
            ),
            'validity_pct': round(100.0 * st_d['validity'] / n, 4),
            'uniqueness_pct': round(
                100.0 * max(0, st_d['rows'] - st_d['duplicates']) / n, 4
            ),
            'duplicate_rows': st_d['duplicates'],
        })
    df_all = pd.DataFrame(rows)
    q_mask = df_all['scope'].isin(['Q1', 'Q2', 'Q3', 'Q4'])
    df_ov = df_all.loc[~q_mask].reset_index(drop=True)
    df_q = df_all.loc[q_mask].sort_values('scope').reset_index(drop=True)
    return df_ov, df_q


def scan_version(
    version_tag: str,
    files: List[Tuple[str, Path]],
    all_cols: List[str],
    compute_dist: bool = False,
) -> Dict[str, Any]:
    """Quet streaming toan bo 1 phien ban va thu thap tat ca thong ke."""
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    meta_rows = sum(pq.ParquetFile(fp).metadata.num_rows for _, fp in files)
    if MAX_BATCHES is not None:
        meta_rows = min(meta_rows, MAX_BATCHES * BATCH_SIZE * len(files))

    track_cols = [c for c in all_cols if c not in ID_TIME_COLS]
    if compute_dist and 'Dist_To_Berth_NM' not in track_cols:
        track_cols.append('Dist_To_Berth_NM')

    trackers: Dict[str, Dict[str, WelfordTracker]] = {
        sc: {c: WelfordTracker(c) for c in track_cols}
        for sc in ('Overall', 'Train', 'Test')
    }
    dama_acc: Dict[str, Dict[str, Any]] = {}
    dedup = SlidingBatchDeduplicator()
    month_cnt, vt_cnt = Counter(), Counter()
    st_cnt, imp_cnt = Counter(), Counter()
    horizon_cnt: Dict[str, Counter] = {
        'Overall': Counter(), 'Train': Counter(), 'Test': Counter()
    }
    sog_edges = np.linspace(0.0, 45.0, 91)
    log_y_edges = np.linspace(0.0, 9.5, 96)
    hist_sog = np.zeros(len(sog_edges) - 1, dtype=np.int64)
    hist_logy = np.zeros(len(log_y_edges) - 1, dtype=np.int64)

    sample_parts: List[pd.DataFrame] = []
    batch_count = 0
    rows_scanned = 0

    for split_lbl, fpath in files:
        pf = pq.ParquetFile(fpath)
        f_rows = int(pf.metadata.num_rows)
        n_take = min(
            max(1, int(round(SAMPLE_SIZE * (f_rows / max(meta_rows, 1))))),
            f_rows,
        )
        sel_pos = np.sort(rng.choice(f_rows, size=n_take, replace=False))
        offset = 0
        pos_idx = 0

        for b_i, batch in enumerate(
            pf.iter_batches(batch_size=BATCH_SIZE, columns=all_cols)
        ):
            if MAX_BATCHES is not None and b_i >= MAX_BATCHES:
                break
            df_b = batch.to_pandas()
            n_b = len(df_b)
            rows_scanned += n_b
            batch_count += 1

            dist_arr = (
                compute_geodesic_dist_nm(df_b)
                if compute_dist
                else pd.to_numeric(
                    df_b['Dist_To_Berth_NM'], errors='coerce'
                ).to_numpy(np.float64)
            )
            if compute_dist:
                df_b['Dist_To_Berth_NM'] = dist_arr.astype(np.float32)

            bdt_s = pd.to_datetime(df_b['base_date_time'], errors='coerce')
            is_q4 = (bdt_s.dt.month >= 10).fillna(False).to_numpy()
            if split_lbl == 'pre_split':
                split_tag = 'Mixed'
            else:
                split_tag = 'Train' if split_lbl == 'train' else 'Test'

            g_keys = ['Overall']
            if split_tag in ('Train', 'Test'):
                g_keys.append(split_tag)
            evaluate_dama_batch(df_b, dist_arr, dedup, dama_acc, g_keys)

            for c in track_cols:
                arr = pd.to_numeric(df_b[c], errors='coerce').to_numpy(
                    np.float64
                )
                trackers['Overall'][c].update(arr)
                if split_tag in ('Train', 'Test'):
                    trackers[split_tag][c].update(arr)
                elif np.any(~is_q4):
                    trackers['Train'][c].update(arr[~is_q4])
                if split_tag == 'Mixed' and np.any(is_q4):
                    trackers['Test'][c].update(arr[is_q4])

            ym = bdt_s.dt.strftime('%Y-%m').fillna('INVALID')
            month_cnt.update(ym.value_counts().to_dict())
            vt_s = pd.to_numeric(df_b['vessel_type'], errors='coerce')
            vt_cnt.update(vt_s.fillna(-1).astype(int).value_counts().to_dict())
            st_s = pd.to_numeric(df_b['status'], errors='coerce')
            st_cnt.update(st_s.fillna(-1).astype(int).value_counts().to_dict())
            if 'imputation_flag' in df_b.columns:
                imp_s = pd.to_numeric(df_b['imputation_flag'], errors='coerce')
                imp_cnt.update(
                    imp_s.fillna(-99).astype(int).value_counts().to_dict()
                )

            sog_v = pd.to_numeric(df_b['sog'], errors='coerce').to_numpy(float)
            sog_v = sog_v[np.isfinite(sog_v)]
            hist_sog += np.histogram(sog_v, bins=sog_edges)[0]

            tgt_v = pd.to_numeric(
                df_b['Target_Hours'], errors='coerce'
            ).to_numpy(float)
            tgt_fin = tgt_v[np.isfinite(tgt_v) & (tgt_v >= 0.0)]
            hist_logy += np.histogram(np.log1p(tgt_fin), bins=log_y_edges)[0]

            bins_h = [
                ('H1_0_24h', (tgt_v > 0.0) & (tgt_v <= 24.0)),
                ('H2_24_48h', (tgt_v > 24.0) & (tgt_v <= 48.0)),
                ('H3_48_72h', (tgt_v > 48.0) & (tgt_v <= 72.0)),
                ('H4_72_168h', (tgt_v > 72.0) & (tgt_v <= 168.0)),
                ('H5_gt_168h', tgt_v > 168.0),
                ('H0_nonpos_or_nan', ~(tgt_v > 0.0)),
            ]
            for h_lbl, h_msk in bins_h:
                cnt_h = int(h_msk.sum())
                horizon_cnt['Overall'][h_lbl] += cnt_h
                if split_tag in ('Train', 'Test'):
                    horizon_cnt[split_tag][h_lbl] += cnt_h

            end = offset + n_b
            right_idx = int(np.searchsorted(sel_pos, end, side='left'))
            if right_idx > pos_idx:
                take_local = sel_pos[pos_idx:right_idx] - offset
                sub = df_b.iloc[take_local].copy()
                sub['_split_src'] = split_tag
                sample_parts.append(sub)
                pos_idx = right_idx
            offset = end

            del df_b
            if batch_count % 20 == 0:
                gc.collect()
        del pf
    gc.collect()

    df_sample = (
        pd.concat(sample_parts, ignore_index=True)
        if sample_parts
        else pd.DataFrame()
    )
    df_ov, df_q = finalize_dama_table(dama_acc, version_tag)
    elapsed = round(time.time() - t0, 2)
    print(
        f'[SCAN {version_tag}] rows={rows_scanned:,} | '
        f'sample={len(df_sample):,} | elapsed={elapsed}s'
    )
    return {
        'version': version_tag,
        'rows_scanned': rows_scanned,
        'elapsed_sec': elapsed,
        'trackers': trackers,
        'dama_overall': df_ov,
        'dama_quarter': df_q,
        'month_cnt': month_cnt,
        'vt_cnt': vt_cnt,
        'st_cnt': st_cnt,
        'imp_cnt': imp_cnt,
        'horizon_cnt': horizon_cnt,
        'sog_edges': sog_edges,
        'hist_sog': hist_sog,
        'log_y_edges': log_y_edges,
        'hist_logy': hist_logy,
        'sample': df_sample,
    }

In [ ]:
"""C05: Chay quet streaming cho RAW & PREP27, xuat DICTIONARY/ va DAMA/."""

raw_res = scan_version('RAW_18', RAW_FILES, COLS_RAW, compute_dist=True)
prep_res = scan_version('PREP_27', PREP_FILES, COLS_PREP27, compute_dist=False)


def build_full_dictionary(res: Dict[str, Any]) -> pd.DataFrame:
    """Gop Welford moments (Overall/Train/Test) voi phan vi tren mau."""
    df_smp = res['sample']
    rows = []
    for scope_name, trk_map in res['trackers'].items():
        if scope_name in ('Train', 'Test') and '_split_src' in df_smp.columns:
            sub_s = df_smp.loc[df_smp['_split_src'] == scope_name]
        else:
            sub_s = df_smp
        for col, trk in trk_map.items():
            info = trk.summary()
            info['version'] = res['version']
            info['scope'] = scope_name
            vals = (
                pd.to_numeric(sub_s[col], errors='coerce').dropna()
                if (not sub_s.empty and col in sub_s.columns)
                else pd.Series(dtype=float)
            )
            qs = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
            q_names = ['P1', 'P5', 'P25', 'P50', 'P75', 'P95', 'P99']
            if len(vals) > 0:
                q_vals = vals.quantile(qs).tolist()
                for qn, qv in zip(q_names, q_vals):
                    info[qn] = round(float(qv), 6)
            else:
                for qn in q_names:
                    info[qn] = None
            rows.append(info)
    return pd.DataFrame(rows)


dict_raw = build_full_dictionary(raw_res)
dict_prep = build_full_dictionary(prep_res)
dict_raw.to_csv(DICT_DIR / 'data_dictionary_raw18_full.csv', index=False)
dict_prep.to_csv(DICT_DIR / 'data_dictionary_prep27_full.csv', index=False)

miss_raw = dict_raw.loc[dict_raw['scope'] == 'Overall', [
    'column', 'missing_count', 'missing_pct'
]].rename(columns={'missing_count': 'raw_miss', 'missing_pct': 'raw_miss_pct'})
miss_prep = dict_prep.loc[dict_prep['scope'] == 'Overall', [
    'column', 'missing_count', 'missing_pct'
]].rename(
    columns={'missing_count': 'prep_miss', 'missing_pct': 'prep_miss_pct'}
)
df_miss_cmp = miss_raw.merge(miss_prep, on='column', how='outer')
df_miss_cmp.to_csv(
    DICT_DIR / 'missing_comparison_raw_vs_prep27.csv', index=False
)

all_months = sorted(
    set(raw_res['month_cnt'].keys()) | set(prep_res['month_cnt'].keys())
)
df_months = pd.DataFrame([
    {
        'month': m,
        'raw_rows': raw_res['month_cnt'].get(m, 0),
        'prep27_rows': prep_res['month_cnt'].get(m, 0),
    }
    for m in all_months
])
df_months.to_csv(DICT_DIR / 'monthly_counts_raw_vs_prep27.csv', index=False)


def _merge_code_counts(
    c_raw: Counter, c_prep: Counter, key_name: str
) -> pd.DataFrame:
    keys = sorted(set(c_raw.keys()) | set(c_prep.keys()))
    raw_tot = max(sum(c_raw.values()), 1)
    prep_tot = max(sum(c_prep.values()), 1)
    df_out = pd.DataFrame([
        {
            key_name: k,
            'raw_count': c_raw.get(k, 0),
            'raw_pct': round(100.0 * c_raw.get(k, 0) / raw_tot, 4),
            'prep27_count': c_prep.get(k, 0),
            'prep27_pct': round(100.0 * c_prep.get(k, 0) / prep_tot, 4),
        }
        for k in keys
    ])
    return df_out.sort_values('raw_count', ascending=False).reset_index(
        drop=True
    )


df_vt = _merge_code_counts(
    raw_res['vt_cnt'], prep_res['vt_cnt'], 'vessel_type'
)
df_st = _merge_code_counts(raw_res['st_cnt'], prep_res['st_cnt'], 'status')
df_vt.to_csv(DICT_DIR / 'vessel_types_raw_vs_prep27.csv', index=False)
df_st.to_csv(DICT_DIR / 'status_raw_vs_prep27.csv', index=False)

imp_tot = max(sum(prep_res['imp_cnt'].values()), 1)
df_imp = pd.DataFrame([
    {
        'imputation_flag': k,
        'count': v,
        'pct': round(100.0 * v / imp_tot, 4),
    }
    for k, v in sorted(prep_res['imp_cnt'].items())
])
df_imp.to_csv(DICT_DIR / 'imputation_flag_prep27.csv', index=False)

# Xuat DAMA 100% tu ket qua chay (KHONG hardcode)
dama_raw_ov = raw_res['dama_overall']
dama_raw_q = raw_res['dama_quarter']
dama_prep_ov = prep_res['dama_overall']
dama_prep_q = prep_res['dama_quarter']

dama_raw_ov.to_csv(DAMA_DIR / 'dama_raw_overall.csv', index=False)
dama_raw_q.to_csv(DAMA_DIR / 'dama_raw_by_quarter.csv', index=False)
dama_prep_ov.to_csv(DAMA_DIR / 'dama_prep27_overall.csv', index=False)
dama_prep_q.to_csv(DAMA_DIR / 'dama_prep27_by_quarter.csv', index=False)

raw_ov_row = dama_raw_ov.loc[dama_raw_ov['scope'] == 'Overall'].iloc[0]
prep_ov_row = dama_prep_ov.loc[dama_prep_ov['scope'] == 'Overall'].iloc[0]


def _sample_spearman_dist_target(df_smp: pd.DataFrame) -> float:
    x = pd.to_numeric(df_smp['Dist_To_Berth_NM'], errors='coerce')
    y = pd.to_numeric(df_smp['Target_Hours'], errors='coerce')
    m = np.isfinite(x) & np.isfinite(y)
    if int(m.sum()) < 3:
        return float('nan')
    return round(float(spearmanr(x[m], y[m])[0]), 5)


raw_rho = _sample_spearman_dist_target(raw_res['sample'])
prep_rho = _sample_spearman_dist_target(prep_res['sample'])

dama_metrics = [
    ('1a. Completeness (Record All)', 'completeness_rec_all_pct'),
    ('1b. Completeness (Record Non-Cargo)', 'completeness_rec_non_cargo_pct'),
    ('1c. Completeness (Field All)', 'completeness_field_all_pct'),
    ('1d. Completeness (Field Non-Cargo)', 'completeness_field_non_cargo_pct'),
    ('2a. Consistency (Kinematic)', 'consistency_kinematic_pct'),
    ('2b. Consistency (Operational <=168h)', 'consistency_operational_pct'),
    ('3. Accuracy (Implied Spd >=1.62kn)', 'accuracy_kinematic_pct'),
    ('4a. Timeliness (2025 Causal)', 'timeliness_2025_causal_pct'),
    ('4b. Timeliness (168h Window)', 'timeliness_168h_window_pct'),
    ('5. Validity (Physical Rules)', 'validity_pct'),
    ('6. Uniqueness (MMSI+Timestamp)', 'uniqueness_pct'),
]
cmp_rows = [
    {
        'dimension': lbl,
        'raw18_pct': float(raw_ov_row[col]),
        'prep27_pct': float(prep_ov_row[col]),
        'delta_pct': round(
            float(prep_ov_row[col]) - float(raw_ov_row[col]), 4
        ),
    }
    for lbl, col in dama_metrics
]
cmp_rows.append({
    'dimension': 'Pearson r (Dist, Target_Hours) [Full]',
    'raw18_pct': float(raw_ov_row['pearson_dist_target_r']),
    'prep27_pct': float(prep_ov_row['pearson_dist_target_r']),
    'delta_pct': round(
        float(prep_ov_row['pearson_dist_target_r'])
        - float(raw_ov_row['pearson_dist_target_r']),
        5,
    ),
})
cmp_rows.append({
    'dimension': 'Spearman rho (Dist, Target_Hours) [Sample]',
    'raw18_pct': raw_rho,
    'prep27_pct': prep_rho,
    'delta_pct': round(prep_rho - raw_rho, 5),
})
df_dama_cmp = pd.DataFrame(cmp_rows)
df_dama_cmp.to_csv(DAMA_DIR / 'dama_comparison_raw_vs_prep27.csv', index=False)

print('=== DAMA 6 CHIEU: RAW_18 VS PREP_27 ===')
display(df_dama_cmp)

## Phần B: EDA Phân Tầng Target (Không Cắt 168h), Tương Quan & Biểu Đồ

- Phân tầng đầy đủ 5 mức `Target_Hours`: `0-24h`, `24-48h`, `48-72h`, `72-168h`, và `> 168h` (tách riêng `Train`, `Test`, `Overall`).
- Đo cả tương quan tuyến tính **Pearson `r`** và tương quan đơn điệu **Spearman `rho`** của toàn bộ 20 đặc trưng mô hình trên `PREP27` đối sánh với các cột tương ứng trên `RAW`.

In [ ]:
"""C07: EDA phan tang Target (khong cat 168h) & Tuong quan Pearson/Spearman."""

h_rows = []
for res in (raw_res, prep_res):
    for scope, cnt in res['horizon_cnt'].items():
        tot = max(sum(cnt.values()), 1)
        for h_bin, h_cnt in sorted(cnt.items()):
            h_rows.append({
                'version': res['version'],
                'scope': scope,
                'horizon': h_bin,
                'count': h_cnt,
                'pct': round(100.0 * h_cnt / tot, 4),
            })
df_horizons = pd.DataFrame(h_rows)
df_horizons.to_csv(
    EDA_DIR / 'target_horizon_breakdown_raw_vs_prep27.csv', index=False
)

df_p_smp = prep_res['sample'].copy()
dist_bins = [0.0, 25.0, 100.0, 250.0, 800.0, np.inf]
dist_lbls = [
    '[0, 25) NM', '[25, 100) NM', '[100, 250) NM',
    '[250, 800) NM', '[800, inf) NM',
]
df_p_smp['dist_band'] = pd.cut(
    pd.to_numeric(df_p_smp['Dist_To_Berth_NM'], errors='coerce'),
    bins=dist_bins,
    labels=dist_lbls,
    right=False,
)
df_band = (
    df_p_smp.groupby('dist_band', observed=False)['Target_Hours']
    .agg(
        count='count',
        mean='mean',
        std='std',
        median='median',
        p90=lambda s: s.quantile(0.90),
        gt_168h_pct=lambda s: 100.0 * float((s > 168.0).mean()),
    )
    .reset_index()
)
df_band.to_csv(EDA_DIR / 'sample_target_by_distance_prep27.csv', index=False)


def _compute_feature_corrs(
    df_smp: pd.DataFrame, feat_cols: List[str], prefix: str
) -> pd.DataFrame:
    y = pd.to_numeric(df_smp[TARGET_COL], errors='coerce').to_numpy(float)
    out_rows = []
    for col in feat_cols:
        if col not in df_smp.columns:
            continue
        x = pd.to_numeric(df_smp[col], errors='coerce').to_numpy(float)
        msk = np.isfinite(x) & np.isfinite(y)
        n_m = int(msk.sum())
        if n_m < 10 or np.std(x[msk]) < 1e-12:
            pr, sr = np.nan, np.nan
        else:
            pr = float(np.corrcoef(x[msk], y[msk])[0, 1])
            sr = float(spearmanr(x[msk], y[msk])[0])
        out_rows.append({
            'feature': col,
            f'{prefix}_n': n_m,
            f'{prefix}_pearson_r': round(pr, 5) if np.isfinite(pr) else None,
            f'{prefix}_spearman_rho': (
                round(sr, 5) if np.isfinite(sr) else None
            ),
        })
    return pd.DataFrame(out_rows)


raw_corr_cols = [c for c in RAW_MODEL_FEATURES_13] + ['Dist_To_Berth_NM']
c_raw_df = _compute_feature_corrs(raw_res['sample'], raw_corr_cols, 'raw18')
c_prep_df = _compute_feature_corrs(
    prep_res['sample'], MODEL_FEATURES_20, 'prep27'
)
df_corr_cmp = c_prep_df.merge(c_raw_df, on='feature', how='outer')
df_corr_cmp['abs_prep27_spearman'] = (
    df_corr_cmp['prep27_spearman_rho'].abs().fillna(-1.0)
)
df_corr_cmp = df_corr_cmp.sort_values(
    'abs_prep27_spearman', ascending=False
).drop(columns=['abs_prep27_spearman'])
df_corr_cmp.to_csv(
    CORR_DIR / 'target_correlations_raw_vs_prep27.csv', index=False
)

corr_mat = df_p_smp[MODEL_FEATURES_20].corr(min_periods=100)
collinear_rows = []
for idx_a, feat_a in enumerate(MODEL_FEATURES_20):
    for feat_b in MODEL_FEATURES_20[idx_a + 1:]:
        r_val = corr_mat.loc[feat_a, feat_b]
        if pd.notna(r_val) and abs(float(r_val)) > 0.85:
            collinear_rows.append({
                'feature_1': feat_a,
                'feature_2': feat_b,
                'pearson_r': round(float(r_val), 5),
            })
df_collinear = pd.DataFrame(
    collinear_rows, columns=['feature_1', 'feature_2', 'pearson_r']
)
df_collinear.to_csv(
    CORR_DIR / 'pearson_collinearity_pairs_prep27.csv', index=False
)

print('=== TOP CORRELATIONS WITH TARGET_HOURS ===')
display(df_corr_cmp.head(15))
display(df_band)

In [ ]:
"""C08: Ve 5 bieu do doi sanh RAW vs PREP27 trong CHARTS/."""


def save_fig(fig: plt.Figure, out_path: Path) -> None:
    """Luu bieu do PNG do phan giai cao va giai phong bo nho."""
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


# 1. DAMA 6 chieu RAW vs PREP27 + Heatmap Q1-Q4 PREP27
dama_bar_cols = [
    ('1a.Comp(All)', 'completeness_rec_all_pct'),
    ('1b.Comp(NoCargo)', 'completeness_rec_non_cargo_pct'),
    ('2a.Cons(Kin)', 'consistency_kinematic_pct'),
    ('2b.Cons(<=168h)', 'consistency_operational_pct'),
    ('3.Accuracy', 'accuracy_kinematic_pct'),
    ('4a.Time(2025)', 'timeliness_2025_causal_pct'),
    ('5.Validity', 'validity_pct'),
    ('6.Uniqueness', 'uniqueness_pct'),
]
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
x_i = np.arange(len(dama_bar_cols))
r_vals = [float(raw_ov_row[c]) for _, c in dama_bar_cols]
p_vals = [float(prep_ov_row[c]) for _, c in dama_bar_cols]
axs[0].bar(x_i - 0.18, r_vals, width=0.36, label='RAW_18', color='#D95F02')
axs[0].bar(x_i + 0.18, p_vals, width=0.36, label='PREP_27', color='#1B9E77')
axs[0].set_xticks(x_i)
axs[0].set_xticklabels([l for l, _ in dama_bar_cols], rotation=28, ha='right')
axs[0].set_ylim(0, 112)
axs[0].set_ylabel('Ty le (%)')
axs[0].set_title('DAMA 6 chieu: RAW_18 vs PREP_27 (Overall)')
axs[0].legend()

q_hm = dama_prep_q[[c for _, c in dama_bar_cols]].to_numpy(dtype=float)
im = axs[1].imshow(q_hm, aspect='auto', cmap='YlGn', vmin=0, vmax=100)
axs[1].set_yticks(np.arange(len(dama_prep_q)))
axs[1].set_yticklabels(dama_prep_q['scope'].tolist())
axs[1].set_xticks(x_i)
axs[1].set_xticklabels([l for l, _ in dama_bar_cols], rotation=28, ha='right')
axs[1].set_title('PREP_27 DAMA theo quy (Q1-Q4)')
fig.colorbar(im, ax=axs[1], shrink=0.85)
save_fig(fig, CHART_DIR / '01_dama_6_dimensions_raw_vs_prep27.png')

# 2. Missing comparison + imputation_flag
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
top_miss = df_miss_cmp.sort_values('raw_miss_pct', ascending=False).head(12)
y_m = np.arange(len(top_miss))
axs[0].barh(
    y_m - 0.18, top_miss['raw_miss_pct'].fillna(0),
    height=0.36, label='RAW_18', color='#D95F02',
)
axs[0].barh(
    y_m + 0.18, top_miss['prep_miss_pct'].fillna(0),
    height=0.36, label='PREP_27', color='#1B9E77',
)
axs[0].set_yticks(y_m)
axs[0].set_yticklabels(top_miss['column'].tolist())
axs[0].set_xlabel('Ty le khuyet (%)')
axs[0].set_title('Ty le khuyet thieu: RAW_18 vs PREP_27')
axs[0].legend()

axs[1].bar(
    df_imp['imputation_flag'].astype(str),
    df_imp['pct'],
    color='#2878B5',
)
axs[1].set_xlabel('imputation_flag')
axs[1].set_ylabel('Ty trong (%)')
axs[1].set_title('Phan bo imputation_flag tren PREP_27')
save_fig(fig, CHART_DIR / '02_missing_and_imputation_flag.png')

# 3. SOG, Vessel Type, Status
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
r_sog_pct = 100.0 * raw_res['hist_sog'] / max(raw_res['hist_sog'].sum(), 1)
p_sog_pct = 100.0 * prep_res['hist_sog'] / max(prep_res['hist_sog'].sum(), 1)
axs[0].stairs(r_sog_pct, raw_res['sog_edges'], label='RAW_18', color='#D95F02')
axs[0].stairs(
    p_sog_pct, prep_res['sog_edges'], label='PREP_27', color='#1B9E77'
)
axs[0].set_xlabel('SOG (knots)')
axs[0].set_ylabel('Ty trong (%)')
axs[0].set_title('Phan bo SOG')
axs[0].legend()

vt_top = df_vt.head(10)
x_vt = np.arange(len(vt_top))
axs[1].bar(x_vt - 0.18, vt_top['raw_pct'], width=0.36, label='RAW_18')
axs[1].bar(x_vt + 0.18, vt_top['prep27_pct'], width=0.36, label='PREP_27')
axs[1].set_xticks(x_vt)
axs[1].set_xticklabels(vt_top['vessel_type'].astype(str), rotation=45)
axs[1].set_title('Top Vessel Type (%)')
axs[1].legend()

st_top = df_st.head(10)
x_st = np.arange(len(st_top))
axs[2].bar(x_st - 0.18, st_top['raw_pct'], width=0.36, label='RAW_18')
axs[2].bar(x_st + 0.18, st_top['prep27_pct'], width=0.36, label='PREP_27')
axs[2].set_xticks(x_st)
axs[2].set_xticklabels(st_top['status'].astype(str), rotation=45)
axs[2].set_title('Top NavStatus (%)')
axs[2].legend()
save_fig(fig, CHART_DIR / '03_sog_vessel_status_raw_vs_prep27.png')

# 4. Full log1p Target distribution + 5 horizons (khong cat 168h)
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
r_ly = 100.0 * raw_res['hist_logy'] / max(raw_res['hist_logy'].sum(), 1)
p_ly = 100.0 * prep_res['hist_logy'] / max(prep_res['hist_logy'].sum(), 1)
axs[0].stairs(r_ly, raw_res['log_y_edges'], label='RAW_18', color='#D95F02')
axs[0].stairs(p_ly, prep_res['log_y_edges'], label='PREP_27', color='#1B9E77')
axs[0].axvline(np.log1p(168.0), color='red', ls='--', label='168h cutoff')
axs[0].set_xlabel('log(1 + Target_Hours)')
axs[0].set_ylabel('Ty trong (%)')
axs[0].set_title('Phan phoi Target_Hours day du (khong cat 168h)')
axs[0].legend()

h_piv = (
    df_horizons.loc[df_horizons['scope'].isin(['Train', 'Test'])]
    .pivot(index='horizon', columns=['version', 'scope'], values='pct')
    .fillna(0.0)
)
h_piv.plot(kind='bar', ax=axs[1])
axs[1].set_ylabel('Ty trong (%)')
axs[1].set_title('Phan tang Target_Hours: Train vs Test (RAW vs PREP27)')
axs[1].tick_params(axis='x', rotation=30)
save_fig(fig, CHART_DIR / '04_target_distribution_full_and_horizons.png')

# 5. Correlation heatmap PREP27 + Target correlation ranking
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
im_c = axs[0].imshow(
    corr_mat.to_numpy(float), cmap='coolwarm', vmin=-1, vmax=1
)
axs[0].set_xticks(np.arange(len(MODEL_FEATURES_20)))
axs[0].set_yticks(np.arange(len(MODEL_FEATURES_20)))
axs[0].set_xticklabels(MODEL_FEATURES_20, rotation=90, fontsize=7)
axs[0].set_yticklabels(MODEL_FEATURES_20, fontsize=7)
axs[0].set_title('Ma tran tuong quan Pearson (20 dac trung PREP27)')
fig.colorbar(im_c, ax=axs[0], shrink=0.8)

top_c = df_corr_cmp.dropna(subset=['prep27_spearman_rho']).head(15)
y_c = np.arange(len(top_c))
axs[1].barh(
    y_c - 0.18, top_c['prep27_pearson_r'].fillna(0),
    height=0.36, label='Pearson r (PREP27)', color='#2878B5',
)
axs[1].barh(
    y_c + 0.18, top_c['prep27_spearman_rho'].fillna(0),
    height=0.36, label='Spearman rho (PREP27)', color='#E69F00',
)
axs[1].set_yticks(y_c)
axs[1].set_yticklabels(top_c['feature'].tolist())
axs[1].axvline(0.0, color='black', lw=0.8)
axs[1].set_title('Top dac trung tuong quan voi Target_Hours')
axs[1].legend()
save_fig(fig, CHART_DIR / '05_correlation_heatmap_and_target_ranking.png')
print('[CHARTS] Da xuat xong 5 bieu do EDA/DAMA/Correlation.')

## Phần C: Kiểm Chứng Cải Thiện CLDL & Đối Chứng Nhanh Mô Hình (RAW vs PREP27)

1. **Mask-and-Recover 3 cấp (`C10`):** Chia tập mẫu sạch theo chuyến (`80%` tham chiếu / `20%` mặt nạ có giao `(mmsi, trip_id)` giữa tham chiếu và mặt nạ) để cả 4 cấp (`trip`, `vessel`, `type`, `group` + `global`) đều hoạt động và đo MAE/RMSE cho `length`, `width`, `draft`.
2. **LightGBM Benchmark (`C11`):** Huấn luyện trên tập Train sạch và chấm trên tập Eval phân tầng chuẩn (`All`, `LE_168h`, `GT_168h`, **tuyệt đối không cắt 168h trên Eval**), so sánh trực tiếp:
   - `RAW_18` (13 đặc trưng thô trên mẫu `RAW`)
   - `PREP_27_RawCols` (13 đặc trưng thô trên mẫu `PREP27`)
   - `PREP_27_Full20` (20 đặc trưng đầy đủ của `PREP27`, loại 4 ID/Time & 2 góc thô)

In [ ]:
"""C10: Kiem chung Mask-and-Recover 3 cap (sua bug C09 khong giao trip)."""

BODY_COLS = ['length', 'width', 'draft']
LEVELS = [
    ('trip', ['mmsi', 'trip_id']),
    ('vessel', ['mmsi']),
    ('type', ['vessel_type']),
    ('group', ['_ship_group']),
]


def _prep_body(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for col in BODY_COLS:
        val = pd.to_numeric(d[col], errors='coerce')
        ok = np.isfinite(val) & (val > 0.0)
        if col == 'draft':
            ok &= val <= 25.5
        d[col] = val.where(ok)
    vt = pd.to_numeric(d['vessel_type'], errors='coerce')
    d['_ship_group'] = pd.Series(pd.NA, index=d.index, dtype='string')
    d.loc[vt.between(70, 79), '_ship_group'] = 'Cargo'
    d.loc[vt.between(80, 89), '_ship_group'] = 'Tanker'
    return d


def fit_body_imputer(ref_df: pd.DataFrame) -> Dict[str, Any]:
    """Hoc bang trung vi 4 cap tu tap tham chieu."""
    ref = _prep_body(ref_df)
    lookup: Dict[str, Any] = {}
    for col in BODY_COLS:
        maps = [
            (lvl, keys, ref.groupby(keys, dropna=True)[col].median())
            for lvl, keys in LEVELS
        ]
        lookup[col] = {'global': float(ref[col].median()), 'maps': maps}
    return lookup


def apply_body_imputer(
    df_target: pd.DataFrame, lookup: Dict[str, Any]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Dien khuyet theo cap uu tien va ghi nhan nguon dien."""
    d = _prep_body(df_target)
    src_rows = []
    for col in BODY_COLS:
        src = pd.Series('observed', index=d.index)
        src.loc[d[col].isna()] = 'unresolved'
        for lvl, keys, mp in lookup[col]['maps']:
            miss = d[col].isna()
            if not miss.any():
                break
            idx_k = pd.MultiIndex.from_frame(d.loc[miss, keys]) if len(
                keys
            ) > 1 else d.loc[miss, keys[0]]
            fill_v = pd.Series(
                mp.reindex(idx_k).to_numpy(), index=d.index[miss]
            )
            hit = fill_v.notna()
            d.loc[miss & hit, col] = fill_v.loc[hit]
            src.loc[miss & hit] = lvl
        still_na = d[col].isna()
        if still_na.any():
            d.loc[still_na, col] = lookup[col]['global']
            src.loc[still_na] = 'global'
        for k_s, cnt_s in src.value_counts().items():
            src_rows.append(
                {'column': col, 'source': k_s, 'count': int(cnt_s)}
            )
    return d, pd.DataFrame(src_rows)


mr_rng = np.random.default_rng(SEED)
mr_pool = _prep_body(prep_res['sample'])
mr_pool = mr_pool.dropna(subset=BODY_COLS).reset_index(drop=True)
perm = mr_rng.permutation(len(mr_pool))
n_ref = int(0.8 * len(mr_pool))
mr_ref = mr_pool.iloc[perm[:n_ref]].copy()
mr_val = mr_pool.iloc[perm[n_ref:]].copy()
mr_lookup = fit_body_imputer(mr_ref)

mr_masked = mr_val.copy()
truth_map: Dict[str, pd.Series] = {}
for col in BODY_COLS:
    m_pos = mr_rng.random(len(mr_masked)) < 0.20
    truth_map[col] = mr_masked.loc[m_pos, col].copy()
    mr_masked.loc[m_pos, col] = np.nan

mr_rec, mr_sources = apply_body_imputer(mr_masked, mr_lookup)
mr_rows = []
for col in BODY_COLS:
    y_t = truth_map[col].to_numpy(float)
    y_h = mr_rec.loc[truth_map[col].index, col].to_numpy(float)
    y_g = np.full(len(y_t), mr_lookup[col]['global'], dtype=float)
    mr_rows.append({
        'column': col,
        'masked_n': len(y_t),
        'hierarchy_MAE': round(float(np.mean(np.abs(y_h - y_t))), 4),
        'hierarchy_RMSE': round(float(np.sqrt(np.mean((y_h - y_t) ** 2))), 4),
        'global_median_MAE': round(float(np.mean(np.abs(y_g - y_t))), 4),
        'global_median_RMSE': round(
            float(np.sqrt(np.mean((y_g - y_t) ** 2))), 4
        ),
    })
df_mr = pd.DataFrame(mr_rows)
df_mr.to_csv(VAL_DIR / 'mask_and_recover_prep27.csv', index=False)
mr_sources.to_csv(VAL_DIR / 'mask_and_recover_sources.csv', index=False)
print('=== MASK-AND-RECOVER 3 CAP ===')
display(df_mr)
display(mr_sources)

In [ ]:
"""C11: Doi chung LightGBM RAW_18 vs PREP_27 tren Eval phan tang chuan."""


def _eval_strata(y_t: np.ndarray, y_p: np.ndarray) -> Dict[str, Any]:
    """Tinh metrics tren 3 tang All / LE_168h / GT_168h."""
    out: Dict[str, Any] = {}
    strata = [
        ('All', np.ones(len(y_t), dtype=bool)),
        ('LE_168h', y_t <= 168.0),
        ('GT_168h', y_t > 168.0),
    ]
    for tag, msk in strata:
        if not np.any(msk):
            out[f'{tag}_N'] = 0
            out[f'{tag}_MAE'] = None
            out[f'{tag}_RMSE'] = None
            out[f'{tag}_MBE'] = None
            continue
        err = y_p[msk] - y_t[msk]
        out[f'{tag}_N'] = int(msk.sum())
        out[f'{tag}_MAE'] = round(float(np.mean(np.abs(err))), 4)
        out[f'{tag}_RMSE'] = round(float(np.sqrt(np.mean(err ** 2))), 4)
        out[f'{tag}_MBE'] = round(float(np.mean(err)), 4)
    return out


def _split_train_eval(
    df_smp: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Tach tap Train va Eval tu mau giu nguyen Eval tho."""
    d = df_smp.copy()
    d[TARGET_COL] = pd.to_numeric(d[TARGET_COL], errors='coerce')
    d = d[np.isfinite(d[TARGET_COL]) & (d[TARGET_COL] > 0.0)].copy()
    if '_split_src' in d.columns and set(d['_split_src']) >= {'Train', 'Test'}:
        tr = d[d['_split_src'] == 'Train'].copy()
        ev = d[d['_split_src'] == 'Test'].copy()
    else:
        bdt = pd.to_datetime(d['base_date_time'], errors='coerce')
        ev_msk = (bdt.dt.month >= 10).fillna(False)
        tr, ev = d.loc[~ev_msk].copy(), d.loc[ev_msk].copy()
    if len(tr) < 100 or len(ev) < 50:
        n_tr = int(0.75 * len(d))
        tr, ev = d.iloc[:n_tr].copy(), d.iloc[n_tr:].copy()
    return tr.reset_index(drop=True), ev.reset_index(drop=True)


def _fit_and_score_lgb(
    df_tr: pd.DataFrame,
    df_ev: pd.DataFrame,
    feat_cols: List[str],
    run_label: str,
) -> Tuple[Dict[str, Any], lgb.LGBMRegressor]:
    """Huan luyen LightGBM tren tap Train va cham Eval phan tang."""
    x_tr = df_tr[feat_cols].apply(pd.to_numeric, errors='coerce')
    y_tr = df_tr[TARGET_COL].to_numpy(float)
    x_ev = df_ev[feat_cols].apply(pd.to_numeric, errors='coerce')
    y_ev = df_ev[TARGET_COL].to_numpy(float)

    n_val = max(1, int(0.15 * len(x_tr)))
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=SEED,
        n_jobs=N_JOBS,
        verbosity=-1,
    )
    model.fit(
        x_tr.iloc[:-n_val],
        y_tr[:-n_val],
        eval_set=[(x_tr.iloc[-n_val:], y_tr[-n_val:])],
        eval_metric='l1',
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    preds = model.predict(x_ev)
    row = {
        'run_label': run_label,
        'n_features': len(feat_cols),
        'train_n': len(x_tr),
        'eval_n': len(x_ev),
        **_eval_strata(y_ev, preds),
    }
    return row, model


raw_tr, raw_ev = _split_train_eval(raw_res['sample'])
prep_tr, prep_ev = _split_train_eval(prep_res['sample'])

res_raw18, _ = _fit_and_score_lgb(
    raw_tr, raw_ev, RAW_MODEL_FEATURES_13, 'RAW18_Sample_13cols'
)
res_prep_rawcols, _ = _fit_and_score_lgb(
    prep_tr, prep_ev, RAW_MODEL_FEATURES_13, 'PREP27_Sample_Raw13cols'
)
res_prep_full, model_prep_full = _fit_and_score_lgb(
    prep_tr, prep_ev, MODEL_FEATURES_20, 'PREP27_Sample_Full20cols'
)

df_lgb_cmp = pd.DataFrame([res_raw18, res_prep_rawcols, res_prep_full])
df_lgb_cmp.to_csv(VAL_DIR / 'lgbm_raw_vs_prep27_comparison.csv', index=False)

gains = model_prep_full.booster_.feature_importance(importance_type='gain')
df_gain = pd.DataFrame({
    'feature': MODEL_FEATURES_20,
    'gain': gains,
    'gain_pct': np.round(100.0 * gains / max(float(gains.sum()), 1.0), 3),
}).sort_values('gain', ascending=False).reset_index(drop=True)
df_gain.to_csv(VAL_DIR / 'lgbm_gain_prep27.csv', index=False)

fig, axs = plt.subplots(1, 2, figsize=(15, 5))
x_l = np.arange(len(df_lgb_cmp))
axs[0].bar(
    x_l - 0.18, df_lgb_cmp['All_MAE'].fillna(0),
    width=0.36, label='All MAE (h)', color='#2878B5',
)
axs[0].bar(
    x_l + 0.18, df_lgb_cmp['LE_168h_MAE'].fillna(0),
    width=0.36, label='LE_168h MAE (h)', color='#1B9E77',
)
axs[0].set_xticks(x_l)
axs[0].set_xticklabels(df_lgb_cmp['run_label'].tolist(), rotation=15)
axs[0].set_ylabel('MAE (hours)')
axs[0].set_title('LightGBM RAW_18 vs PREP_27 (All & LE_168h)')
axs[0].legend()

top_g = df_gain.head(15).iloc[::-1]
axs[1].barh(top_g['feature'], top_g['gain_pct'], color='#E69F00')
axs[1].set_xlabel('Feature Gain (%)')
axs[1].set_title('Top 15 Feature Importance (PREP27 Full20)')
save_fig(fig, CHART_DIR / '06_lgbm_raw_vs_prep27_and_gain.png')

print('=== LIGHTGBM RAW_18 VS PREP_27 ===')
display(df_lgb_cmp)
display(df_gain.head(15))

In [ ]:
"""C12: Xuat run_manifest.json, README_results.txt va dong goi ZIP."""

manifest = {
    'raw_mode': RAW_MODE,
    'raw_rows_scanned': raw_res['rows_scanned'],
    'prep27_mode': PREP_MODE,
    'prep27_rows_scanned': prep_res['rows_scanned'],
    'batch_size': BATCH_SIZE,
    'sample_size': SAMPLE_SIZE,
    'max_batches': MAX_BATCHES,
    'prep27_model_features_20': MODEL_FEATURES_20,
    'versions': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'lightgbm': lgb.__version__,
    },
}
(OUT / 'run_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

readme_txt = (
    'KET QUA DS317_02 (RAW_18 VS PREP_27)\n'
    f'RAW_18 ({RAW_MODE}): {raw_res["rows_scanned"]:,} dong\n'
    f'PREP_27 ({PREP_MODE}): {prep_res["rows_scanned"]:,} dong\n'
    'Thu muc xuat: DICTIONARY/, DAMA/, EDA/, CORRELATION/, '
    'VALIDATION/, CHARTS/\n'
)
(OUT / 'README_results.txt').write_text(readme_txt, encoding='utf-8')

zip_path = (
    Path('/kaggle/working/ds317_02_eda_dama_results.zip')
    if Path('/kaggle/working').exists()
    else OUT.parent / 'ds317_02_eda_dama_results.zip'
)
n_packed = 0
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUT.rglob('*')):
        if fpath.is_file() and fpath.suffix.lower() != '.parquet':
            zf.write(fpath, arcname=str(fpath.relative_to(OUT)))
            n_packed += 1

print(f'[DONE] Da dong goi {n_packed} tep -> {zip_path}')
if zip_path.exists():
    try:
        display(FileLink(str(zip_path)))
    except Exception:
        pass